# Preprocesamiento ENE 2024
## Inserción Laboral de Migrantes en Chile
**Fuente:** Encuesta Nacional de Empleo (ENE) 2024 — INE Chile
**Objetivo:** Limpiar, recodificar y construir variables analíticas para el análisis de brecha laboral migrante/chileno.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Rutas relativas al notebook
BASE = os.path.abspath(os.path.join('..', '..'))
RAW_ENE = os.path.join(BASE, 'input', 'data', 'ene_2024.csv')
OUT_DATA = os.path.join(BASE, 'output', 'data')
os.makedirs(OUT_DATA, exist_ok=True)

print(f"Ruta datos crudos: {RAW_ENE}")
print(f"Ruta salida: {OUT_DATA}")

## 1. Carga de datos

In [ ]:
df_raw = pd.read_csv(RAW_ENE, encoding='latin-1', sep=None, engine='python')
print(f"Dimensiones ENE cruda: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
df_raw.head(3)

## 2. Exploración inicial

In [ ]:
# Tipos y nulos
info = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'nulos': df_raw.isnull().sum(),
    'pct_nulo': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'n_unicos': df_raw.nunique()
})
print("Variables con nulos significativos (>5%):")
print(info[info['pct_nulo'] > 5][['dtype','nulos','pct_nulo']].to_string())

In [ ]:
# Variables de migración disponibles
print("mig1 (nacimiento/residencia):", df_raw['mig1'].value_counts().to_dict())
print()
print("nacionalidad (top 10):", df_raw['nacionalidad'].value_counts().head(10).to_dict())
print()
print("Nota: código 152 = Chile (ISO 3166-1)")

In [ ]:
# Variables laborales clave
print("activ (1=Ocupado, 2=Desocupado, 3=Inactivo):", df_raw['activ'].value_counts().to_dict())
print()
print("ocup_form (1=Formal, 2=Informal):", df_raw['ocup_form'].value_counts().to_dict())
print()
print("cine11_1d (nivel edu ISCED):", df_raw['cine11_1d'].value_counts().sort_index().to_dict())
print()
print("cae_general (grupo ocupacional CIUO):", df_raw['cae_general'].value_counts().sort_index().to_dict())

## 3. Filtrado de población en edad de trabajar

In [ ]:
# Filtrar 15 años y más (edad de trabajar según INE Chile)
df = df_raw[df_raw['edad'] >= 15].copy()
print(f"Población 15+ años: {len(df):,} registros ({len(df)/len(df_raw)*100:.1f}% del total)")

## 4. Variable de condición migratoria

In [ ]:
# Definición: migrante = persona cuya nacionalidad no es chilena (152)
# Se prefiere nacionalidad sobre país de nacimiento porque captura la condición jurídica actual
df['migrante'] = (df['nacionalidad'] != 152).astype(int)

# Verificar con mig1 (1=nació en Chile): migrante si mig1 != 1 y no es código especial
mig1_migrante = df['mig1'].isin([2, 3, 4]).astype(int)

print("Migrantes por nacionalidad:", df['migrante'].value_counts().to_dict())
print("Migrantes por mig1:", mig1_migrante.value_counts().to_dict())
print()
print("Consistencia entre definiciones:")
consistencia = pd.crosstab(df['migrante'], mig1_migrante,
                            rownames=['nac≠152'], colnames=['mig1∈{2,3,4}'])
print(consistencia)
print()
print("Se usa nacionalidad como definidor principal por mayor cobertura y precisión jurídica.")

In [ ]:
# País de origen de los migrantes (mig2_cod para país de origen)
# Distribución de migrantes por tipo según mig1
mig_tipo = df[df['migrante'] == 1]['mig1'].value_counts()
print("Migrantes por tipo (mig1):")
print("  2 = +5 años en Chile:", mig_tipo.get(2, 0))
print("  3 = 1-4 años en Chile:", mig_tipo.get(3, 0))
print("  4 = < 1 año en Chile:", mig_tipo.get(4, 0))
print("  Total migrantes:", df['migrante'].sum())
print(f"  Tasa migratoria en muestra: {df['migrante'].mean()*100:.1f}%")

## 5. Limpieza de variables laborales

In [ ]:
# --- Factor de expansión anual (usa coma como decimal) ---
df['ponderador'] = (
    df['fact_anual']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)
print("Ponderador (fact_anual) - estadísticas:")
print(df['ponderador'].describe())

In [ ]:
# --- Horas trabajadas ---
# habituales: horas habituales. Valores >168 son inválidos (hay 999)
df['horas_habituales'] = df['habituales'].copy()
df.loc[df['horas_habituales'] > 168, 'horas_habituales'] = np.nan
df.loc[df['horas_habituales'] <= 0, 'horas_habituales'] = np.nan

print("Horas habituales antes de limpieza:")
print(df_raw[df_raw['edad']>=15]['habituales'].describe())
print()
print("Horas habituales después de limpieza:")
print(df['horas_habituales'].describe())

In [ ]:
# --- Condición de actividad ---
# activ: 1=Ocupado, 2=Desocupado, 3=Inactivo
df['ocupado']   = (df['activ'] == 1).astype(int)
df['desocupado']= (df['activ'] == 2).astype(int)
df['inactivo']  = (df['activ'] == 3).astype(int)

print("Condición de actividad (filas):")
print(pd.DataFrame({
    'Ocupado':    [df['ocupado'].sum()],
    'Desocupado': [df['desocupado'].sum()],
    'Inactivo':   [df['inactivo'].sum()]
}).to_string(index=False))

In [ ]:
# --- Formalidad (solo para ocupados) ---
# ocup_form: 1=Formal, 2=Informal
df['formal'] = np.nan
mask_ocup = df['activ'] == 1
df.loc[mask_ocup & (df['ocup_form'] == 1), 'formal'] = 1
df.loc[mask_ocup & (df['ocup_form'] == 2), 'formal'] = 0
df['formal'] = df['formal'].astype('Int64')  # permite NaN

print("Formalidad (solo ocupados):")
print(df['formal'].value_counts(dropna=False).to_dict())
print(f"Tasa de formalidad: {df['formal'].mean()*100:.1f}%")

## 6. Recodificación de variables categóricas

In [ ]:
# --- Nivel educativo (CINE 2011, adaptado a nomenclatura chilena) ---
# 0=Sin estudios, 1=Básica, 2=Media incompleta, 3=Media completa,
# 4=Técnica post-media, 5=Técnica superior (CFT/IP), 6=Universitaria/Postgrado, 9=Sin clasificar
edu_map = {0: 'Sin estudios', 1: 'Básica', 2: 'Media incomp.',
           3: 'Media comp.', 4: 'Téc. post-media', 5: 'Téc. superior', 6: 'Universitaria', 9: 'Sin clasificar'}
df['edu_nivel'] = df['cine11_1d'].map(edu_map)
df['edu_nivel'] = pd.Categorical(df['edu_nivel'],
    categories=['Sin estudios','Básica','Media incomp.','Media comp.','Téc. post-media','Téc. superior','Universitaria','Sin clasificar'],
    ordered=True)

# Nivel educativo agrupado (3 grupos)
df['edu_grupo'] = pd.cut(
    df['cine11_1d'],
    bins=[-1, 1.5, 3.5, 9],
    labels=['Básico (0-1)', 'Medio (2-3)', 'Superior (4-6)']
)
df.loc[df['cine11_1d'] == 9, 'edu_grupo'] = np.nan

print("Distribución educativa:")
print(df['edu_nivel'].value_counts().to_string())

In [ ]:
# --- Grupo ocupacional (CIUO-88 adaptado por INE) ---
# cae_general: 0=No aplica, 1=Sin clasificar, 2=Directivos, 3=Profesionales,
# 4=Técnicos, 5=Admin, 6=Servicios, 7=Agropecuarios, 8=Artesanos/operarios, 9=Elementales
ocup_map = {
    0: 'No_ocup', 1: 'Sin_clasif', 2: 'Directivos', 3: 'Profesionales',
    4: 'Tecnicos', 5: 'Admin', 6: 'Servicios', 7: 'Agro', 8: 'Artesanos', 9: 'Elementales'
}
df['ocup_grupo'] = df['cae_general'].map(ocup_map)

# Ocupación calificada: grupos 2-5 (directivos, profesionales, técnicos, admin)
# Ocupación no calificada: grupos 6-9
df['ocup_calificada'] = np.where(
    df['cae_general'].isin([2, 3, 4, 5]), 1,
    np.where(df['cae_general'].isin([6, 7, 8, 9]), 0, np.nan)
)
df['ocup_calificada'] = df['ocup_calificada'].astype('Int64')

print("Grupos ocupacionales:")
print(df['ocup_grupo'].value_counts().to_string())

In [ ]:
# --- Sector económico ---
# sector: 1=Público, 2=Privado, 3=Doméstico
sector_map = {1: 'Público', 2: 'Privado', 3: 'Doméstico'}
df['sector_eco'] = df['sector'].map(sector_map)

# --- Categoría ocupacional ---
# 0=No aplica, 1=Empleador, 2=Cuenta_propia, 3=Dependiente, 4=Doméstico, 5=Familiar
cat_map = {0:'No_aplica',1:'Empleador',2:'Cuenta_propia',3:'Dependiente',
           4:'Doméstico',5:'Familiar_no_rem',6:'Aprendiz',7:'Otro'}
df['cat_ocupacion'] = df['categoria_ocupacion'].map(cat_map)

print("Sector económico:")
print(df['sector_eco'].value_counts().to_string())
print()
print("Categoría ocupacional:")
print(df['cat_ocupacion'].value_counts().to_string())

## 7. Feature engineering: variables derivadas

In [ ]:
# --- Grupo de edad ---
df['grupo_edad'] = pd.cut(
    df['edad'],
    bins=[14, 24, 34, 44, 54, 64, 200],
    labels=['15-24', '25-34', '35-44', '45-54', '55-64', '65+']
)

# --- Sexo ---
df['sexo_str'] = df['sexo'].map({1: 'Hombre', 2: 'Mujer'})

# --- Región (macrozona) ---
macrozona_map = {
    1:'Norte_grande', 2:'Norte_grande', 15:'Norte_grande',
    3:'Norte_chico',  4:'Norte_chico',
    5:'Centro',       13:'Centro',
    6:'Sur',          7:'Sur',       16:'Sur',
    8:'Sur',          9:'Sur',       14:'Sur',
    10:'Austral',     11:'Austral',  12:'Austral'
}
df['macrozona'] = df['region'].map(macrozona_map)

print("Grupos de edad:")
print(df['grupo_edad'].value_counts().sort_index().to_string())
print()
print("Macrozona:")
print(df['macrozona'].value_counts().to_string())

In [ ]:
# --- Subempleo por insuficiencia de horas (obe) ---
# obe: 1=con subempleo por insuficiencia de horas, 0=sin subempleo
df['subempleo_horas'] = df['obe'].map({1: 1, 0: 0})

# --- Tiempo parcial involuntario (tpi) ---
df['tiempo_parcial_inv'] = df['tpi'].map({1: 1, 0: 0})

# --- Sobrecalificación: persona con edu superior en ocupación elemental ---
edu_superior = df['cine11_1d'].isin([4, 5, 6])
ocup_elemental = df['cae_general'] == 9
df['sobrecalificado'] = (edu_superior & ocup_elemental & (df['activ'] == 1)).astype(int)
# Solo aplica para ocupados con educación disponible
df.loc[df['activ'] != 1, 'sobrecalificado'] = np.nan
df['sobrecalificado'] = df['sobrecalificado'].astype('Int64')

print("Subempleo por horas (ocupados):", df[df['activ']==1]['subempleo_horas'].value_counts().to_dict())
print("Sobrecalificados:", df['sobrecalificado'].sum(), "de", (df['activ']==1).sum(), "ocupados")
print(f"Tasa sobrecalificación entre ocupados: {df['sobrecalificado'].mean()*100:.1f}%")

## 8. Selección de variables finales y validación

In [ ]:
# Variables finales para análisis
VARS_FINALES = [
    # Identificadores y ponderador
    'idrph', 'ponderador',
    # Sociodemográficas
    'sexo', 'sexo_str', 'edad', 'grupo_edad', 'region', 'macrozona',
    # Migración
    'migrante', 'mig1', 'nacionalidad',
    # Educación
    'cine11_1d', 'edu_nivel', 'edu_grupo',
    # Actividad
    'activ', 'ocupado', 'desocupado', 'inactivo',
    # Laborales (ocupados)
    'horas_habituales', 'formal', 'ocup_form',
    'cae_general', 'ocup_grupo', 'ocup_calificada',
    'sector', 'sector_eco',
    'categoria_ocupacion', 'cat_ocupacion',
    'ftp', 'obe', 'tpi', 'subempleo_horas', 'tiempo_parcial_inv',
    'sobrecalificado',
]

df_final = df[VARS_FINALES].copy()
print(f"Dataset ENE procesado: {df_final.shape[0]:,} filas × {df_final.shape[1]} columnas")
print()
print("Resumen de nulos en variables clave:")
nulos_clave = df_final[['migrante','sexo','edad','activ','formal','horas_habituales']].isnull().sum()
print(nulos_clave.to_string())

In [ ]:
# Validación: distribución por condición migratoria
print("=== RESUMEN POR CONDICIÓN MIGRATORIA ===")
resumen = df_final.groupby('migrante').agg(
    n_total=('activ', 'count'),
    ocupados=('ocupado', 'sum'),
    tasa_ocupacion=('ocupado', 'mean'),
    formales=('formal', lambda x: x.sum(skipna=True)),
    tasa_formalidad=('formal', lambda x: x.mean(skipna=True))
).round(3)
resumen.index = ['Chilenos', 'Migrantes']
print(resumen.to_string())

## 9. Guardado del dataset procesado

In [ ]:
out_path = os.path.join(OUT_DATA, 'ene_procesada.parquet')
df_final.to_parquet(out_path, index=False)
print(f"ENE procesada guardada en: {out_path}")
print(f"Tamaño: {os.path.getsize(out_path)/1024:.0f} KB")

# También guardar CSV para portabilidad
out_csv = os.path.join(OUT_DATA, 'ene_procesada.csv')
df_final.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f"ENE procesada también guardada en CSV: {out_csv}")

In [ ]:
# Verificación final
df_check = pd.read_parquet(out_path)
print("Verificación de lectura:")
print(df_check.shape)
print(df_check[['migrante','ocupado','formal']].describe().round(3))